In [0]:
source_dir = "/Volumes/my_catalog/my_volume/my_file_volume/source_crm/"
schema_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_schemas/sales_details_bronze/"
checkpoint_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_checkpoints/sales_details_bronze/"
target_table = "my_catalog.bronze.sales_details"

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_dir)
      .option("cloudFiles.includeExistingFiles", "true")  
      .option("header", "true")
      .option("pathGlobFilter", "sales_details.csv")            
      .load(source_dir)
)

(df.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_dir)
 .option("mergeSchema", "true")  # Allow schema evolution to resolve mismatch
 .outputMode("append")
 .trigger(once=True)                                     
 .toTable(target_table)
)